# Project - Extensions: experiments after the milestone

| **Student Name** | **Hemanth Kumar Reddyvari** |
|---|---|
| Student ID | a3185100 |
| Course Name | Programming for Artificial Intelligence and Machine Learning 2 |

In this notebook I am doing the experiments added after the milestone, with the same pipeline, settings and models as `AIML Workout Final.ipynb`. That notebook is not changed and its results (`results_final`) are not retrained.

Before running, change runtime to GPU: Runtime -> Change runtime type -> T4 GPU. Then run the sections in order (you need `kaggle.json` for the data download).

| Exp | Question | Models | Approx. T4 time |
|---|---|---|---|
| A. Same input | Does the CNN's larger gap survive when CNN and features get the same input? (milestone feedback) | CNN on raw windows, features from standardised windows | 25 min |
| B. Blocked split | How much of the random-split gap is window overlap, and how much is seeing the person? | all wrist models, CNN and features on leg and pocket | 10 min |
| C. Augmentation | Does rotation augmentation narrow the gap, especially for the pocket? | CNN with rotation + jitter | 25 min |
| D. Sensor fusion | Does wrist + leg in one model recover both arm and leg exercises? | CNN on 14 channels | 15 min |
| E. Transfer | Does pre-training on the leg help the pocket when pocket data is scarce? | CNN, pre-trained vs from scratch | 30 min |
| App | Does the app hold up on all 10 people, and do the new set checks catch rest logged as exercise? | app, one held-out CNN per person | 25 min |

Every run is logged to `results_ext/results_ext_log.csv` as soon as it finishes, so a disconnect only loses the run in progress. Each experiment cell can be run on its own after sections 1 to 5.

**Run order:** 1 Setup, 2 Imports, 3 Data, 4 Models, 5 Evaluation, then any of A to E and 11 (App), then 12 Summary and 13 Download.

## 1. Setup
Use Drive (`True`) so the cache and results survive a disconnect. Needs `kaggle.json`.

In [23]:
from google.colab import drive, files

var_usedrive_bool = True  # True = save to Google Drive (recommended for this notebook)

if var_usedrive_bool:
    drive.mount('/content/drive')

import os
if not os.path.exists('/content/recgym/RecGym.csv'):
    files.upload()  # upload kaggle.json
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d zhaxidelebsz/10-gym-exercises-with-615-abstracted-features -p /content
    !unzip -q -o /content/10-gym-exercises-with-615-abstracted-features.zip -d /content/recgym
!ls -lh /content/recgym

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 454M
-rw-r--r-- 1 root root 454M Mar  6  2025 RecGym.csv


## 2. Imports and settings
Same settings as the final notebook. New results go to `results_ext`, separate from `results_final`.

In [24]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
# TensorFlow is imported inside train_keras, after torch has used the GPU

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

var_device_obj = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", var_device_obj)

if var_usedrive_bool:
    assert os.path.ismount('/content/drive'), "Drive not mounted, run Setup first"
    var_basedir_txt = "/content/drive/MyDrive/paiml2"
else:
    var_basedir_txt = "/content/paiml2"
print("Saving to:", var_basedir_txt)

var_csvpath_txt = "/content/recgym/RecGym.csv"
var_cachedir_txt = f"{var_basedir_txt}/cache"
var_resultsdir_txt = f"{var_basedir_txt}/results_ext"
var_figuresdir_txt = f"{var_basedir_txt}/figures_ext"
var_logpath_txt = f"{var_resultsdir_txt}/results_ext_log.csv"

for var_folder_txt in [var_cachedir_txt, var_resultsdir_txt, var_figuresdir_txt]:
    os.makedirs(var_folder_txt, exist_ok=True)

# settings, same as the final notebook
var_seed_int = 42
var_window_int = 80       # 4 s at 20 Hz
var_stride_int = 40       # 2 s overlap
var_purity_float = 0.9    # drop windows that mix two activities
var_channels_list = ['A_x', 'A_y', 'A_z', 'G_x', 'G_y', 'G_z', 'C_1']
var_epochs_int = 40
var_batch_int = 256
var_lr_float = 1e-3

# new settings
var_blocksize_int = 50        # blocked split: 50 windows = about 100 s of recording per block
var_jitter_float = 0.05       # augmentation: Gaussian noise added after standardisation
var_finetune_fracs_list = [0.10, 0.25]  # transfer: share of target-position training data available

Device: cuda
Saving to: /content/drive/MyDrive/paiml2


## 3. Data
`build_windows` is identical to the final notebook (same cache file). `build_fused_windows` is new: it lines up the wrist and leg recordings of each session and stacks them into 14 channels.

In [25]:
def build_windows(var_position_txt):
    var_cachepath_txt = f"{var_cachedir_txt}/{var_position_txt}_w{var_window_int}_s{var_stride_int}.npz"
    if os.path.exists(var_cachepath_txt):
        print("Loading cached windows:", var_cachepath_txt)
        var_cache_npz = np.load(var_cachepath_txt, allow_pickle=True)
        return (var_cache_npz['X'], var_cache_npz['y'],
                var_cache_npz['subjects'], list(var_cache_npz['labels']))

    print(f"Building windows for {var_position_txt} (slow, runs once)")
    assert os.path.exists(var_csvpath_txt), "RecGym.csv missing, run Setup"
    var_columns_list = ['Subject', 'Session', 'Position', 'Workout'] + var_channels_list
    var_data_df = pd.read_csv(var_csvpath_txt, usecols=var_columns_list)
    var_data_df = var_data_df[var_data_df['Position'] == var_position_txt]

    var_labels_list = sorted(var_data_df['Workout'].unique())
    var_labelmap_dict = {var_label_txt: i for i, var_label_txt in enumerate(var_labels_list)}

    var_x_list, var_y_list, var_subjects_list = [], [], []
    for (var_subject_int, var_session_int), var_group_df in var_data_df.groupby(['Subject', 'Session'], sort=False):
        var_signal_arr = var_group_df[var_channels_list].to_numpy(dtype=np.float32)
        var_label_arr = var_group_df['Workout'].map(var_labelmap_dict).to_numpy()
        for var_start_int in range(0, len(var_signal_arr) - var_window_int + 1, var_stride_int):
            var_counts_arr = np.bincount(var_label_arr[var_start_int:var_start_int + var_window_int],
                                         minlength=len(var_labels_list))
            var_main_int = var_counts_arr.argmax()
            if var_counts_arr[var_main_int] < var_purity_float * var_window_int:
                continue
            var_x_list.append(var_signal_arr[var_start_int:var_start_int + var_window_int])
            var_y_list.append(var_main_int)
            var_subjects_list.append(var_subject_int)

    var_x_arr, var_y_arr, var_subjects_arr = np.stack(var_x_list), np.array(var_y_list), np.array(var_subjects_list)
    np.savez_compressed(var_cachepath_txt, X=var_x_arr, y=var_y_arr, subjects=var_subjects_arr,
                        labels=np.array(var_labels_list, dtype=object))
    print("Cached to", var_cachepath_txt)
    return var_x_arr, var_y_arr, var_subjects_arr, var_labels_list


def build_fused_windows(var_positions_list=("wrist", "leg")):
    # wrist and leg rows of the same session, lined up row by row, 14 channels per window
    var_name_txt = "_".join(var_positions_list)
    var_cachepath_txt = f"{var_cachedir_txt}/fused_{var_name_txt}_w{var_window_int}_s{var_stride_int}.npz"
    if os.path.exists(var_cachepath_txt):
        print("Loading cached windows:", var_cachepath_txt)
        var_cache_npz = np.load(var_cachepath_txt, allow_pickle=True)
        return (var_cache_npz['X'], var_cache_npz['y'],
                var_cache_npz['subjects'], list(var_cache_npz['labels']))

    print(f"Building fused windows for {var_name_txt} (slow, runs once)")
    var_columns_list = ['Subject', 'Session', 'Position', 'Workout'] + var_channels_list
    var_data_df = pd.read_csv(var_csvpath_txt, usecols=var_columns_list)
    var_data_df = var_data_df[var_data_df['Position'].isin(var_positions_list)]
    var_labels_list = sorted(var_data_df['Workout'].unique())
    var_labelmap_dict = {var_label_txt: i for i, var_label_txt in enumerate(var_labels_list)}

    var_x_list, var_y_list, var_subjects_list, var_check_list = [], [], [], []
    for (var_subject_int, var_session_int), var_group_df in var_data_df.groupby(['Subject', 'Session'], sort=False):
        var_parts_list = [var_group_df[var_group_df['Position'] == p] for p in var_positions_list]
        var_len_int = min(len(p) for p in var_parts_list)
        var_lab_list = [p['Workout'].map(var_labelmap_dict).to_numpy()[:var_len_int] for p in var_parts_list]
        var_agree_float = float((var_lab_list[0] == var_lab_list[1]).mean()) if var_len_int else 0.0
        var_check_list.append((var_subject_int, var_session_int, [len(p) for p in var_parts_list], round(var_agree_float, 4)))
        if var_agree_float < 0.99:
            continue  # the two recordings do not line up, skip the session
        var_signal_arr = np.concatenate(
            [p[var_channels_list].to_numpy(dtype=np.float32)[:var_len_int] for p in var_parts_list], axis=1)
        for var_start_int in range(0, var_len_int - var_window_int + 1, var_stride_int):
            var_counts_list = [np.bincount(l[var_start_int:var_start_int + var_window_int], minlength=len(var_labels_list))
                               for l in var_lab_list]
            var_main_int = var_counts_list[0].argmax()
            # pure in both recordings, same activity
            if var_counts_list[1].argmax() != var_main_int or \
               min(c[var_main_int] for c in var_counts_list) < var_purity_float * var_window_int:
                continue
            var_x_list.append(var_signal_arr[var_start_int:var_start_int + var_window_int])
            var_y_list.append(var_main_int)
            var_subjects_list.append(var_subject_int)

    var_check_df = pd.DataFrame(var_check_list, columns=['subject', 'session', 'rows_per_position', 'label_agreement'])
    print("Alignment check (label agreement between positions, row by row):")
    print(var_check_df.to_string(index=False))
    print(f"{(var_check_df['label_agreement'] >= 0.99).sum()} of {len(var_check_df)} sessions line up and are used")
    assert var_x_list, "No session lines up between positions, fusion is not possible this way"

    var_x_arr, var_y_arr, var_subjects_arr = np.stack(var_x_list), np.array(var_y_list), np.array(var_subjects_list)
    np.savez_compressed(var_cachepath_txt, X=var_x_arr, y=var_y_arr, subjects=var_subjects_arr,
                        labels=np.array(var_labels_list, dtype=object))
    print("Cached to", var_cachepath_txt)
    return var_x_arr, var_y_arr, var_subjects_arr, var_labels_list


def standardise_windows(var_x_arr):
    # each channel to mean 0, std 1 within its own window
    var_mean_arr = var_x_arr.mean(axis=1, keepdims=True)
    var_std_arr = var_x_arr.std(axis=1, keepdims=True) + 1e-6
    return ((var_x_arr - var_mean_arr) / var_std_arr).astype(np.float32)


def extract_features(var_x_arr):
    # 7 stats x channels, same as the final notebook
    var_features_list = [
        var_x_arr.mean(axis=1),
        var_x_arr.std(axis=1),
        var_x_arr.min(axis=1),
        var_x_arr.max(axis=1),
        np.percentile(var_x_arr, 25, axis=1),
        np.percentile(var_x_arr, 75, axis=1),
        np.abs(np.diff(var_x_arr, axis=1)).mean(axis=1),
    ]
    return np.concatenate(var_features_list, axis=1).astype(np.float32)


def make_data_dict(var_name_txt, var_x_arr, var_y_arr, var_subjects_arr, var_labels_list):
    print(f"{var_name_txt}: {var_x_arr.shape[0]:,} windows, {var_x_arr.shape[2]} channels, {len(var_labels_list)} classes")
    var_xstd_arr = standardise_windows(var_x_arr)
    return {
        "position": var_name_txt,
        "x_raw": var_x_arr.astype(np.float32),        # augmentation, CNN on raw input
        "x_std": var_xstd_arr,                         # CNN, LSTM
        "features": extract_features(var_x_arr),       # Keras, as in the final notebook
        "features_std": extract_features(var_xstd_arr),  # Keras on the CNN's input (Exp A)
        "y": var_y_arr,
        "subjects": var_subjects_arr,
        "labels": var_labels_list,
        "n_cls": len(var_labels_list),
        "n_ch": var_x_arr.shape[2],
    }


var_data_cache_dict = {}

def load_position(var_position_txt):
    if var_position_txt not in var_data_cache_dict:
        if var_position_txt.startswith("fused"):
            var_parts_tuple = build_fused_windows(("wrist", "leg"))
        else:
            var_parts_tuple = build_windows(var_position_txt)
        var_data_cache_dict[var_position_txt] = make_data_dict(var_position_txt, *var_parts_tuple)
    return var_data_cache_dict[var_position_txt]

## 4. Models
Same CNN, LSTM and Keras network as the final notebook. Training is split into `fit_torch` and `predict_torch` so the same code serves augmentation and transfer learning.

**Augmentation:** a random 3D rotation is applied to the acceleration and gyroscope triplets of each training window (the same rotation for both, capacitance untouched), then the window is standardised, then small Gaussian jitter is added. Scaling augmentation is left out on purpose: per-window standardisation would undo it. Because the published signals are min-max scaled per channel, the rotation is an approximation of a physical rotation.

In [4]:
def class_weights(var_y_arr, var_ncls_int):
    var_counts_arr = np.bincount(var_y_arr, minlength=var_ncls_int).astype(np.float32)
    return var_counts_arr.sum() / (var_ncls_int * np.maximum(var_counts_arr, 1))


class CNN(nn.Module):
    def __init__(self, var_nch_int, var_ncls_int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(var_nch_int, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(256, var_ncls_int)

    def forward(self, var_x_tensor):
        var_x_tensor = var_x_tensor.transpose(1, 2)
        var_x_tensor = self.net(var_x_tensor).squeeze(-1)
        return self.head(self.drop(var_x_tensor))


class LSTMNet(nn.Module):
    def __init__(self, var_nch_int, var_ncls_int, var_hidden_int=128):
        super().__init__()
        self.lstm = nn.LSTM(var_nch_int, var_hidden_int, num_layers=2, batch_first=True, dropout=0.3)
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(var_hidden_int, var_ncls_int)

    def forward(self, var_x_tensor):
        var_out_tensor, _ = self.lstm(var_x_tensor)
        return self.head(self.drop(var_out_tensor.mean(dim=1)))


def random_rotations(var_n_int, var_maxdeg_float):
    # Rodrigues' formula: random axis, angle uniform in [-max, max]
    var_axis_tensor = torch.randn(var_n_int, 3, device=var_device_obj)
    var_axis_tensor = var_axis_tensor / var_axis_tensor.norm(dim=1, keepdim=True)
    var_theta_tensor = (torch.rand(var_n_int, device=var_device_obj) * 2 - 1) * np.deg2rad(var_maxdeg_float)
    var_k_tensor = torch.zeros(var_n_int, 3, 3, device=var_device_obj)
    var_k_tensor[:, 0, 1], var_k_tensor[:, 0, 2] = -var_axis_tensor[:, 2], var_axis_tensor[:, 1]
    var_k_tensor[:, 1, 0], var_k_tensor[:, 1, 2] = var_axis_tensor[:, 2], -var_axis_tensor[:, 0]
    var_k_tensor[:, 2, 0], var_k_tensor[:, 2, 1] = -var_axis_tensor[:, 1], var_axis_tensor[:, 0]
    var_sin_tensor = torch.sin(var_theta_tensor)[:, None, None]
    var_cos_tensor = torch.cos(var_theta_tensor)[:, None, None]
    var_eye_tensor = torch.eye(3, device=var_device_obj).expand(var_n_int, 3, 3)
    return var_eye_tensor + var_sin_tensor * var_k_tensor + (1 - var_cos_tensor) * (var_k_tensor @ var_k_tensor)


def augment_batch(var_xb_tensor, var_maxdeg_float):
    # var_xb_tensor: RAW windows (batch, time, channels); every 7 channels are one sensor: A_xyz, G_xyz, C_1
    var_out_tensor = var_xb_tensor.clone()
    var_r_tensor = random_rotations(var_xb_tensor.shape[0], var_maxdeg_float)
    for var_base_int in range(0, var_xb_tensor.shape[2], 7):  # works for 7 or 14 channels
        for var_off_int in (0, 3):  # acceleration, then gyroscope, same rotation
            var_sl = slice(var_base_int + var_off_int, var_base_int + var_off_int + 3)
            var_out_tensor[:, :, var_sl] = var_xb_tensor[:, :, var_sl] @ var_r_tensor.transpose(1, 2)
    var_mean_tensor = var_out_tensor.mean(dim=1, keepdim=True)
    var_std_tensor = var_out_tensor.std(dim=1, unbiased=False, keepdim=True) + 1e-6
    var_out_tensor = (var_out_tensor - var_mean_tensor) / var_std_tensor
    return var_out_tensor + var_jitter_float * torch.randn_like(var_out_tensor)


def fit_torch(var_modelclass, var_xtrain_arr, var_ytrain_arr, var_ncls_int, var_epochs=var_epochs_int,
              var_maxdeg_float=None, var_init_model_obj=None):
    # var_maxdeg_float=None: no augmentation, var_xtrain_arr is already model input
    # var_maxdeg_float=number: var_xtrain_arr is RAW, rotated + standardised + jittered per batch
    torch.manual_seed(var_seed_int)
    var_weights_tensor = torch.tensor(class_weights(var_ytrain_arr, var_ncls_int)).to(var_device_obj)
    var_loader_obj = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(var_xtrain_arr), torch.tensor(var_ytrain_arr)),
        batch_size=var_batch_int, shuffle=True)
    var_model_obj = var_init_model_obj if var_init_model_obj is not None else \
        var_modelclass(var_xtrain_arr.shape[2], var_ncls_int).to(var_device_obj)
    var_optimizer_obj = torch.optim.Adam(var_model_obj.parameters(), lr=var_lr_float)
    var_scheduler_obj = torch.optim.lr_scheduler.CosineAnnealingLR(var_optimizer_obj, T_max=var_epochs)
    var_lossfn_obj = nn.CrossEntropyLoss(weight=var_weights_tensor)
    for var_epoch_int in range(var_epochs):
        var_model_obj.train()
        for var_xb_tensor, var_yb_tensor in var_loader_obj:
            var_xb_tensor, var_yb_tensor = var_xb_tensor.to(var_device_obj), var_yb_tensor.to(var_device_obj)
            if var_maxdeg_float is not None:
                var_xb_tensor = augment_batch(var_xb_tensor, var_maxdeg_float)
            var_optimizer_obj.zero_grad()
            var_lossfn_obj(var_model_obj(var_xb_tensor), var_yb_tensor).backward()
            if isinstance(var_model_obj, LSTMNet):
                nn.utils.clip_grad_norm_(var_model_obj.parameters(), 1.0)
            var_optimizer_obj.step()
        var_scheduler_obj.step()
    return var_model_obj


def predict_torch(var_model_obj, var_x_arr):
    var_model_obj.eval()
    var_x_tensor = torch.tensor(var_x_arr)
    var_preds_list = []
    with torch.no_grad():
        for i in range(0, len(var_x_tensor), var_batch_int):
            var_preds_list.append(var_model_obj(var_x_tensor[i:i + var_batch_int].to(var_device_obj)).argmax(1).cpu())
    return torch.cat(var_preds_list).numpy()


# ---- train functions: (data dict, train indices, test indices) -> predictions ----

def train_cnn(var_data_dict, var_train_arr, var_test_arr):
    # same as the final notebook: per-window standardised input
    var_x_arr, var_y_arr = var_data_dict['x_std'], var_data_dict['y']
    var_model_obj = fit_torch(CNN, var_x_arr[var_train_arr], var_y_arr[var_train_arr], var_data_dict['n_cls'])
    return predict_torch(var_model_obj, var_x_arr[var_test_arr])


def train_lstm(var_data_dict, var_train_arr, var_test_arr):
    var_x_arr, var_y_arr = var_data_dict['x_std'], var_data_dict['y']
    var_model_obj = fit_torch(LSTMNet, var_x_arr[var_train_arr], var_y_arr[var_train_arr], var_data_dict['n_cls'])
    return predict_torch(var_model_obj, var_x_arr[var_test_arr])


def train_cnn_raw(var_data_dict, var_train_arr, var_test_arr):
    # Exp A: CNN on the SAME raw windows the feature model uses.
    # Each channel is z-scored with mean and std from the training fold (like the feature model's scaler),
    # not per window, so level and spread information is kept.
    var_x_arr, var_y_arr = var_data_dict['x_raw'], var_data_dict['y']
    var_mean_arr = var_x_arr[var_train_arr].mean(axis=(0, 1), keepdims=True)
    var_std_arr = var_x_arr[var_train_arr].std(axis=(0, 1), keepdims=True) + 1e-6
    var_scaled_arr = ((var_x_arr - var_mean_arr) / var_std_arr).astype(np.float32)
    var_model_obj = fit_torch(CNN, var_scaled_arr[var_train_arr], var_y_arr[var_train_arr], var_data_dict['n_cls'])
    return predict_torch(var_model_obj, var_scaled_arr[var_test_arr])


def make_train_cnn_aug(var_maxdeg_float):
    def train_cnn_aug(var_data_dict, var_train_arr, var_test_arr):
        # Exp C: train on RAW windows augmented per batch; test on normal standardised windows
        var_model_obj = fit_torch(CNN, var_data_dict['x_raw'][var_train_arr], var_data_dict['y'][var_train_arr],
                                  var_data_dict['n_cls'], var_maxdeg_float=var_maxdeg_float)
        return predict_torch(var_model_obj, var_data_dict['x_std'][var_test_arr])
    return train_cnn_aug


def _keras(var_f_arr, var_y_arr, var_ncls_int, var_train_arr, var_test_arr):
    import tensorflow as tf
    for var_gpu_obj in tf.config.list_physical_devices('GPU'):
        try:
            tf.config.experimental.set_memory_growth(var_gpu_obj, True)
        except RuntimeError:
            pass
    tf.keras.backend.clear_session()
    tf.random.set_seed(var_seed_int)
    var_scaler_obj = StandardScaler().fit(var_f_arr[var_train_arr])
    var_weights_arr = class_weights(var_y_arr[var_train_arr], var_ncls_int)
    var_model_obj = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(var_f_arr.shape[1],)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(var_ncls_int, activation='softmax'),
    ])
    var_model_obj.compile(optimizer=tf.keras.optimizers.Adam(var_lr_float), loss='sparse_categorical_crossentropy')
    var_model_obj.fit(var_scaler_obj.transform(var_f_arr[var_train_arr]), var_y_arr[var_train_arr],
                      epochs=var_epochs_int, batch_size=var_batch_int, verbose=0,
                      class_weight={i: float(w) for i, w in enumerate(var_weights_arr)})
    return var_model_obj.predict(var_scaler_obj.transform(var_f_arr[var_test_arr]),
                                 batch_size=1024, verbose=0).argmax(axis=1)


def train_keras(var_data_dict, var_train_arr, var_test_arr):
    # same as the final notebook: features from raw windows
    return _keras(var_data_dict['features'], var_data_dict['y'], var_data_dict['n_cls'], var_train_arr, var_test_arr)


def train_keras_std(var_data_dict, var_train_arr, var_test_arr):
    # Exp A: features from the per-window standardised windows the CNN sees
    # (mean and std columns become constant and carry nothing, the other 5 stats keep shape information)
    return _keras(var_data_dict['features_std'], var_data_dict['y'], var_data_dict['n_cls'], var_train_arr, var_test_arr)


var_models_dict = {
    "CNN": train_cnn,
    "LSTM": train_lstm,
    "Keras-Features": train_keras,
    "CNN-RawInput": train_cnn_raw,
    "Keras-StdFeatures": train_keras_std,
}

## 5. Evaluation
Same protocols as the final notebook, plus a **blocked random split**: consecutive windows are grouped into blocks of 50 (about 100 s), 20% of blocks go to test, and any training window next to a test window is removed. No training window then overlaps a test window, but the same people are still in both sets. So:

- random minus blocked = what window overlap adds,
- blocked minus LOSO = what knowing the person adds.

In [5]:
def random_split(var_n_int):
    var_perm_arr = np.random.default_rng(var_seed_int).permutation(var_n_int)
    var_cut_int = int(0.8 * var_n_int)
    return var_perm_arr[:var_cut_int], var_perm_arr[var_cut_int:]


def blocked_split(var_n_int):
    # windows are stored in recording order (subject, session, time), so neighbours in the array overlap in time
    var_blockid_arr = np.arange(var_n_int) // var_blocksize_int
    var_nblocks_int = var_blockid_arr.max() + 1
    var_testblocks_arr = np.random.default_rng(var_seed_int).permutation(var_nblocks_int)[:int(0.2 * var_nblocks_int)]
    var_istest_arr = np.isin(var_blockid_arr, var_testblocks_arr)
    var_neartest_arr = var_istest_arr.copy()
    var_neartest_arr[1:] |= var_istest_arr[:-1]   # window just after a test window
    var_neartest_arr[:-1] |= var_istest_arr[1:]   # window just before a test window
    return np.where(~var_neartest_arr)[0], np.where(var_istest_arr)[0]


def score(var_true_arr, var_pred_arr, var_ncls_int):
    return (accuracy_score(var_true_arr, var_pred_arr),
            f1_score(var_true_arr, var_pred_arr, average='macro', zero_division=0),
            f1_score(var_true_arr, var_pred_arr, average=None, labels=range(var_ncls_int), zero_division=0))


def append_log(var_row_dict):
    var_row_dict["timestamp"] = pd.Timestamp.now().isoformat(timespec='seconds')
    var_log_df = pd.DataFrame([var_row_dict])
    if os.path.exists(var_logpath_txt):
        var_log_df = pd.concat([pd.read_csv(var_logpath_txt), var_log_df], ignore_index=True)
    var_log_df.to_csv(var_logpath_txt, index=False)


def run_protocols(var_data_dict, var_modelname_txt, var_experiment_txt, var_protocols_tuple=("random", "loso"),
                  var_train_fn=None):
    var_position_txt = var_data_dict['position']
    var_y_arr, var_subjects_arr, var_ncls_int = var_data_dict['y'], var_data_dict['subjects'], var_data_dict['n_cls']
    var_train_fn = var_train_fn or var_models_dict[var_modelname_txt]
    print(f"\n{'=' * 60}\n{var_experiment_txt}: {var_position_txt} / {var_modelname_txt}\n{'=' * 60}")
    var_row_dict = {"experiment": var_experiment_txt, "position": var_position_txt, "model": var_modelname_txt,
                    "epochs": var_epochs_int, "lr": var_lr_float}
    var_save_dict = {"labels": np.array(var_data_dict['labels'], dtype=object)}

    for var_protocol_txt in [p for p in var_protocols_tuple if p in ("random", "blocked")]:
        var_train_arr, var_test_arr = (random_split if var_protocol_txt == "random" else blocked_split)(len(var_y_arr))
        var_pred_arr = var_train_fn(var_data_dict, var_train_arr, var_test_arr)
        var_acc_float, var_f1_float, var_pc_arr = score(var_y_arr[var_test_arr], var_pred_arr, var_ncls_int)
        print(f"  {var_protocol_txt:<8} split  n_train={len(var_train_arr):>6}  acc {var_acc_float:.3f}  macroF1 {var_f1_float:.3f}")
        var_row_dict.update({f"{var_protocol_txt}_acc": round(var_acc_float, 4),
                             f"{var_protocol_txt}_macro_f1": round(var_f1_float, 4)})
        var_save_dict[f"per_class_{var_protocol_txt}"] = var_pc_arr

    if "loso" in var_protocols_tuple:
        var_subject_list = sorted(np.unique(var_subjects_arr))
        var_acc_list, var_f1_list, var_pc_list, var_t_list, var_p_list = [], [], [], [], []
        for var_subject_int in var_subject_list:
            var_train_arr = np.where(var_subjects_arr != var_subject_int)[0]
            var_test_arr = np.where(var_subjects_arr == var_subject_int)[0]
            var_pred_arr = var_train_fn(var_data_dict, var_train_arr, var_test_arr)
            var_acc_float, var_f1_float, var_pc_arr = score(var_y_arr[var_test_arr], var_pred_arr, var_ncls_int)
            var_acc_list.append(var_acc_float); var_f1_list.append(var_f1_float); var_pc_list.append(var_pc_arr)
            var_t_list.append(var_y_arr[var_test_arr]); var_p_list.append(var_pred_arr)
            print(f"  subject {var_subject_int:>2}  n={len(var_test_arr):>5}  acc {var_acc_float:.3f}  macroF1 {var_f1_float:.3f}")
        var_f1_arr = np.array(var_f1_list)
        print(f"  LOSO mean      acc {np.mean(var_acc_list):.3f}  macroF1 {var_f1_arr.mean():.3f} +/- {var_f1_arr.std():.3f}")
        var_row_dict.update({"loso_acc_mean": round(float(np.mean(var_acc_list)), 4),
                             "loso_macro_f1_mean": round(float(var_f1_arr.mean()), 4),
                             "loso_macro_f1_sd": round(float(var_f1_arr.std()), 4)})
        var_save_dict.update({"per_class_loso": np.array(var_pc_list).mean(axis=0), "loso_f1": var_f1_arr,
                              "loso_acc": np.array(var_acc_list), "subjects": np.array(var_subject_list),
                              "cm_loso": confusion_matrix(np.concatenate(var_t_list), np.concatenate(var_p_list),
                                                          labels=range(var_ncls_int))})
        if "random_macro_f1" in var_row_dict:
            var_row_dict["gap_macro_f1"] = round(var_row_dict["random_macro_f1"] - var_row_dict["loso_macro_f1_mean"], 4)
            print(f"  protocol gap   {var_row_dict['gap_macro_f1']:+.3f}")

    np.savez_compressed(f"{var_resultsdir_txt}/{var_experiment_txt}_{var_position_txt}_{var_modelname_txt}.npz",
                        **var_save_dict)
    append_log(var_row_dict)
    print(f"  logged {var_experiment_txt} {var_position_txt}/{var_modelname_txt}")

## 6. Experiment A: same input for both approaches
Milestone feedback: the feature model read raw windows while the CNN read per-window standardised windows, so the comparison mixed architecture and input. Two fixes, both on the wrist under both protocols:
- **CNN-RawInput:** the CNN on the raw windows, scaled per channel with training-fold statistics, like the feature model.
- **Keras-StdFeatures:** the feature model on features computed from the standardised windows the CNN sees.

If the CNN's gap stays larger than the feature model's when both see the same input, the milestone finding is about the architecture, not the input. Leg and pocket are optional (uncomment).

In [6]:
for var_position_txt in ["wrist"]:  # optional: ["wrist", "leg", "pocket"]
    var_data_dict = load_position(var_position_txt)
    for var_modelname_txt in ["CNN-RawInput", "Keras-StdFeatures"]:  # torch first, then TF
        run_protocols(var_data_dict, var_modelname_txt, "A_sameinput")

Loading cached windows: /content/drive/MyDrive/paiml2/cache/wrist_w80_s40.npz
wrist: 36,328 windows, 7 channels, 12 classes

A_sameinput: wrist / CNN-RawInput
  random   split  n_train= 29062  acc 0.974  macroF1 0.965
  subject  1  n= 3388  acc 0.956  macroF1 0.949
  subject  2  n= 4360  acc 0.953  macroF1 0.922
  subject  3  n= 3591  acc 0.977  macroF1 0.965
  subject  4  n= 3786  acc 0.926  macroF1 0.899
  subject  5  n= 3551  acc 0.960  macroF1 0.959
  subject  6  n= 4106  acc 0.931  macroF1 0.911
  subject  7  n= 3775  acc 0.948  macroF1 0.933
  subject  8  n= 3338  acc 0.973  macroF1 0.971
  subject  9  n= 3177  acc 0.956  macroF1 0.957
  subject 10  n= 3256  acc 0.977  macroF1 0.965
  LOSO mean      acc 0.956  macroF1 0.943 +/- 0.024
  protocol gap   +0.022
  logged A_sameinput wrist/CNN-RawInput

A_sameinput: wrist / Keras-StdFeatures
  random   split  n_train= 29062  acc 0.933  macroF1 0.926
  subject  1  n= 3388  acc 0.921  macroF1 0.904


  subject  2  n= 4360  acc 0.933  macroF1 0.908
  subject  3  n= 3591  acc 0.935  macroF1 0.907
  subject  4  n= 3786  acc 0.872  macroF1 0.852
  subject  5  n= 3551  acc 0.939  macroF1 0.925
  subject  6  n= 4106  acc 0.939  macroF1 0.909
  subject  7  n= 3775  acc 0.913  macroF1 0.890
  subject  8  n= 3338  acc 0.909  macroF1 0.916
  subject  9  n= 3177  acc 0.923  macroF1 0.912
  subject 10  n= 3256  acc 0.937  macroF1 0.921
  LOSO mean      acc 0.922  macroF1 0.904 +/- 0.020
  protocol gap   +0.022
  logged A_sameinput wrist/Keras-StdFeatures


## 7. Experiment B: blocked random split
Random-split score with no window overlap between training and test, for every model and position from the milestone. Only one model is trained per row (no LOSO), and the random and LOSO numbers come from `results_final`.

In [7]:
for var_position_txt, var_modelnames_list in [("wrist", ["CNN", "LSTM", "Keras-Features"]),
                                             ("leg", ["CNN", "Keras-Features"]),
                                             ("pocket", ["CNN", "Keras-Features"])]:
    var_data_dict = load_position(var_position_txt)
    for var_modelname_txt in var_modelnames_list:  # torch models listed before Keras
        run_protocols(var_data_dict, var_modelname_txt, "B_blocked", var_protocols_tuple=("blocked",))


B_blocked: wrist / CNN
  blocked  split  n_train= 28848  acc 0.970  macroF1 0.958
  logged B_blocked wrist/CNN

B_blocked: wrist / LSTM
  blocked  split  n_train= 28848  acc 0.954  macroF1 0.936
  logged B_blocked wrist/LSTM

B_blocked: wrist / Keras-Features
  blocked  split  n_train= 28848  acc 0.949  macroF1 0.932
  logged B_blocked wrist/Keras-Features
Loading cached windows: /content/drive/MyDrive/paiml2/cache/leg_w80_s40.npz
leg: 33,092 windows, 7 channels, 12 classes

B_blocked: leg / CNN
  blocked  split  n_train= 26270  acc 0.955  macroF1 0.922
  logged B_blocked leg/CNN

B_blocked: leg / Keras-Features
  blocked  split  n_train= 26270  acc 0.784  macroF1 0.766
  logged B_blocked leg/Keras-Features
Loading cached windows: /content/drive/MyDrive/paiml2/cache/pocket_w80_s40.npz
pocket: 35,707 windows, 7 channels, 12 classes

B_blocked: pocket / CNN
  blocked  split  n_train= 28361  acc 0.891  macroF1 0.825
  logged B_blocked pocket/CNN

B_blocked: pocket / Keras-Features
  bloc

## 8. Experiment C: rotation augmentation
The pocket CNN fell from 0.946 to 0.620 on new people, which I explained by phone orientation varying between people. If that is right, training with random rotations should help the pocket most.
- pocket: rotations up to 180 degrees (any orientation in the pocket)
- wrist: rotations up to 30 degrees (a watch can only sit a little differently)

Compare with the unaugmented CNN in `results_final`. Differences under about 0.02 are within run-to-run variation.

In [8]:
for var_position_txt, var_maxdeg_float in [("pocket", 180.0), ("wrist", 30.0)]:
    var_data_dict = load_position(var_position_txt)
    run_protocols(var_data_dict, f"CNN-Rot{int(var_maxdeg_float)}", "C_augment",
                  var_train_fn=make_train_cnn_aug(var_maxdeg_float))


C_augment: pocket / CNN-Rot180
  random   split  n_train= 28565  acc 0.866  macroF1 0.841
  subject  1  n= 3367  acc 0.786  macroF1 0.688
  subject  2  n= 4403  acc 0.861  macroF1 0.758
  subject  3  n= 3509  acc 0.823  macroF1 0.772
  subject  4  n= 3484  acc 0.779  macroF1 0.733
  subject  5  n= 3489  acc 0.805  macroF1 0.743
  subject  6  n= 4033  acc 0.796  macroF1 0.679
  subject  7  n= 3586  acc 0.769  macroF1 0.696
  subject  8  n= 3343  acc 0.846  macroF1 0.808
  subject  9  n= 3190  acc 0.743  macroF1 0.621
  subject 10  n= 3303  acc 0.833  macroF1 0.776
  LOSO mean      acc 0.804  macroF1 0.727 +/- 0.053
  protocol gap   +0.114
  logged C_augment pocket/CNN-Rot180

C_augment: wrist / CNN-Rot30
  random   split  n_train= 29062  acc 0.977  macroF1 0.971
  subject  1  n= 3388  acc 0.949  macroF1 0.925
  subject  2  n= 4360  acc 0.938  macroF1 0.907
  subject  3  n= 3591  acc 0.963  macroF1 0.953
  subject  4  n= 3786  acc 0.933  macroF1 0.902
  subject  5  n= 3551  acc 0.963  m

## 9. Experiment D: sensor fusion (wrist + leg)
Each sensor recognises its own limb, so one CNN that sees both should cover arm and leg exercises. The two recordings are lined up row by row inside each session; the cell prints how well they agree and skips any session that does not line up (label agreement below 99%).

**Check the alignment printout.** If few sessions line up, tell me before using the result.

In [26]:
# ---- Experiment D, version 2: line up wrist and leg by the time shift that matches their labels best ----
var_maxlag_int = 6000        # search shifts of up to 6000 rows (5 minutes) either way
var_minagree_float = 0.95    # use a session only if the labels agree on at least 95% of rows after the shift


def agreement(var_a_arr, var_b_arr, var_lag_int):
    # lag > 0: leg starts var_lag_int rows later than wrist
    if var_lag_int >= 0:
        var_a_arr, var_b_arr = var_a_arr[var_lag_int:], var_b_arr
    else:
        var_a_arr, var_b_arr = var_a_arr, var_b_arr[-var_lag_int:]
    var_n_int = min(len(var_a_arr), len(var_b_arr))
    if var_n_int == 0:
        return 0.0, 0
    return float((var_a_arr[:var_n_int] == var_b_arr[:var_n_int]).mean()), var_n_int


def best_lag(var_a_arr, var_b_arr):
    var_minoverlap_int = int(0.5 * min(len(var_a_arr), len(var_b_arr)))
    def ok(var_lag_int):
        var_agree_float, var_n_int = agreement(var_a_arr, var_b_arr, var_lag_int)
        return var_agree_float if var_n_int >= var_minoverlap_int else -1
    var_coarse_int = max(range(-var_maxlag_int, var_maxlag_int + 1, 10), key=ok)          # every 0.5 s
    return max(range(var_coarse_int - 10, var_coarse_int + 11), key=ok)                    # then row by row


def build_fused_windows(var_positions_list=("wrist", "leg")):
    var_name_txt = "_".join(var_positions_list)
    var_cachepath_txt = f"{var_cachedir_txt}/fused_{var_name_txt}_lag_w{var_window_int}_s{var_stride_int}.npz"
    if os.path.exists(var_cachepath_txt):
        print("Loading cached windows:", var_cachepath_txt)
        var_cache_npz = np.load(var_cachepath_txt, allow_pickle=True)
        return (var_cache_npz['X'], var_cache_npz['y'],
                var_cache_npz['subjects'], list(var_cache_npz['labels']))

    print(f"Building fused windows for {var_name_txt} with time-shift alignment (slow, runs once)")
    var_columns_list = ['Subject', 'Session', 'Position', 'Workout'] + var_channels_list
    var_data_df = pd.read_csv(var_csvpath_txt, usecols=var_columns_list)
    var_data_df = var_data_df[var_data_df['Position'].isin(var_positions_list)]
    var_labels_list = sorted(var_data_df['Workout'].unique())
    var_labelmap_dict = {var_label_txt: i for i, var_label_txt in enumerate(var_labels_list)}

    var_x_list, var_y_list, var_subjects_list, var_check_list = [], [], [], []
    for (var_subject_int, var_session_int), var_group_df in var_data_df.groupby(['Subject', 'Session'], sort=False):
        var_w_df = var_group_df[var_group_df['Position'] == var_positions_list[0]]
        var_l_df = var_group_df[var_group_df['Position'] == var_positions_list[1]]
        if len(var_w_df) == 0 or len(var_l_df) == 0:
            var_check_list.append((var_subject_int, var_session_int, len(var_w_df), len(var_l_df), None, None, 0.0, "missing position"))
            continue
        var_wlab_arr = var_w_df['Workout'].map(var_labelmap_dict).to_numpy()
        var_llab_arr = var_l_df['Workout'].map(var_labelmap_dict).to_numpy()
        var_before_float = agreement(var_wlab_arr, var_llab_arr, 0)[0]
        var_lag_int = best_lag(var_wlab_arr, var_llab_arr)
        var_after_float, var_n_int = agreement(var_wlab_arr, var_llab_arr, var_lag_int)
        var_used_bool = var_after_float >= var_minagree_float
        var_check_list.append((var_subject_int, var_session_int, len(var_w_df), len(var_l_df), var_lag_int / 20,
                               round(var_before_float, 4), round(var_after_float, 4), "used" if var_used_bool else "skipped"))
        if not var_used_bool:
            continue
        var_ws_int, var_ls_int = (var_lag_int, 0) if var_lag_int >= 0 else (0, -var_lag_int)
        var_wsig_arr = var_w_df[var_channels_list].to_numpy(dtype=np.float32)[var_ws_int:var_ws_int + var_n_int]
        var_lsig_arr = var_l_df[var_channels_list].to_numpy(dtype=np.float32)[var_ls_int:var_ls_int + var_n_int]
        var_signal_arr = np.concatenate([var_wsig_arr, var_lsig_arr], axis=1)
        var_lab_list = [var_wlab_arr[var_ws_int:var_ws_int + var_n_int], var_llab_arr[var_ls_int:var_ls_int + var_n_int]]
        for var_start_int in range(0, var_n_int - var_window_int + 1, var_stride_int):
            var_counts_list = [np.bincount(l[var_start_int:var_start_int + var_window_int], minlength=len(var_labels_list))
                               for l in var_lab_list]
            var_main_int = var_counts_list[0].argmax()
            # pure in both recordings and the same activity, so leftover misalignment cannot mislabel a window
            if var_counts_list[1].argmax() != var_main_int or \
               min(c[var_main_int] for c in var_counts_list) < var_purity_float * var_window_int:
                continue
            var_x_list.append(var_signal_arr[var_start_int:var_start_int + var_window_int])
            var_y_list.append(var_main_int)
            var_subjects_list.append(var_subject_int)

    var_check_df = pd.DataFrame(var_check_list, columns=['subject', 'session', 'wrist_rows', 'leg_rows', 'shift_s',
                                                         'agree_before', 'agree_after', 'status'])
    var_check_df.to_csv(f"{var_resultsdir_txt}/D_fusion_alignment.csv", index=False)
    print(var_check_df.to_string(index=False))
    print(f"\n{(var_check_df['status'] == 'used').sum()} of {len(var_check_df)} sessions line up and are used; "
          f"people with at least one session: {sorted(set(var_subjects_list))}")
    if len(set(var_subjects_list)) < 10:
        return None  # not every person has an aligned session, so 10-fold LOSO is not possible

    var_x_arr, var_y_arr, var_subjects_arr = np.stack(var_x_list), np.array(var_y_list), np.array(var_subjects_list)
    np.savez_compressed(var_cachepath_txt, X=var_x_arr, y=var_y_arr, subjects=var_subjects_arr,
                        labels=np.array(var_labels_list, dtype=object))
    print("Cached to", var_cachepath_txt)
    return var_x_arr, var_y_arr, var_subjects_arr, var_labels_list


# build the aligned windows, then three CNNs on exactly the same windows:
# both sensors (14 channels), wrist channels only, leg channels only
var_fused_tuple = build_fused_windows(("wrist", "leg"))
if var_fused_tuple is None:
    print("\nFUSION SKIPPED: the wrist and leg recordings are not time-aligned in RecGym.")
    print("Too few sessions line up for 10-fold LOSO (see the table above, saved as D_fusion_alignment.csv).")
    print("Reported as a data limitation; fusion needs synchronised sensors (future work).")
else:
    var_fx_arr, var_fy_arr, var_fsubjects_arr, var_flabels_list = var_fused_tuple
    var_fused_dict = make_data_dict("fused_wrist_leg", var_fx_arr, var_fy_arr, var_fsubjects_arr, var_flabels_list)
    var_wristonly_dict = make_data_dict("fused_set_wrist_only", var_fx_arr[:, :, :7], var_fy_arr, var_fsubjects_arr, var_flabels_list)
    var_legonly_dict = make_data_dict("fused_set_leg_only", var_fx_arr[:, :, 7:], var_fy_arr, var_fsubjects_arr, var_flabels_list)

    run_protocols(var_fused_dict, "CNN", "D_fusion")
    run_protocols(var_wristonly_dict, "CNN", "D_fusion", var_protocols_tuple=("loso",))
    run_protocols(var_legonly_dict, "CNN", "D_fusion", var_protocols_tuple=("loso",))

    # per-class LOSO F1 on the same windows
    var_pc_df = pd.DataFrame({
        var_n_txt: np.load(f"{var_resultsdir_txt}/D_fusion_{var_n_txt}_CNN.npz", allow_pickle=True)['per_class_loso']
        for var_n_txt in ["fused_set_wrist_only", "fused_set_leg_only", "fused_wrist_leg"]}, index=var_flabels_list)
    var_pc_df.columns = ["wrist only", "leg only", "wrist + leg"]
    var_pc_df["fusion minus best single"] = var_pc_df["wrist + leg"] - var_pc_df[["wrist only", "leg only"]].max(axis=1)
    print("\nPer-class LOSO F1, same windows for all three\n" + var_pc_df.round(3).to_string())

Building fused windows for wrist_leg with time-shift alignment (slow, runs once)
 subject  session  wrist_rows  leg_rows  shift_s  agree_before  agree_after           status
       1        1       51880     52920   -32.00        0.6816       0.7075          skipped
       1        2       25680         0      NaN           NaN       0.0000 missing position
       1        3       30120     30320     0.45        0.9420       0.9453          skipped
       1        4       22560     22160     3.75        0.7775       0.8114          skipped
       1        5       23040     22280    11.95        0.5790       0.8128          skipped
       2        1       58800     58520    -4.10        0.8686       0.9202          skipped
       2        2       36680     36280     6.60        0.8168       0.8801          skipped
       2        3       38960     38440     5.25        0.8653       0.9614             used
       2        4       30520     31560    -8.55        0.6129       0.8071       

## 10. Experiment E: cross-position transfer learning
Can a CNN pre-trained on the leg help the pocket when little pocket data is available?
For each held-out person (LOSO): pre-train on the leg windows of the other 9, then fine-tune on 10% or 25% of their pocket windows (taken as whole blocks of recording), and compare with a CNN trained from scratch on the same pocket data. The held-out person is never used for pre-training or fine-tuning.

In [12]:
def take_fraction(var_idx_arr, var_frac_float, var_seed2_int):
    # whole blocks of consecutive windows, like having recorded fewer sessions
    var_blocks_arr = var_idx_arr // var_blocksize_int
    var_unique_arr = np.unique(var_blocks_arr)
    var_keep_arr = np.random.default_rng(var_seed2_int).permutation(var_unique_arr)[:max(1, int(round(var_frac_float * len(var_unique_arr))))]
    return var_idx_arr[np.isin(var_blocks_arr, var_keep_arr)]


def run_transfer(var_source_txt="leg", var_target_txt="pocket"):
    var_src_dict, var_tgt_dict = load_position(var_source_txt), load_position(var_target_txt)
    assert var_src_dict['labels'] == var_tgt_dict['labels'], "label sets differ between positions"
    var_ncls_int = var_tgt_dict['n_cls']
    var_rows_list = []
    for var_subject_int in sorted(np.unique(var_tgt_dict['subjects'])):
        var_srctrain_arr = np.where(var_src_dict['subjects'] != var_subject_int)[0]
        var_tgtpool_arr = np.where(var_tgt_dict['subjects'] != var_subject_int)[0]
        var_test_arr = np.where(var_tgt_dict['subjects'] == var_subject_int)[0]
        var_ytest_arr = var_tgt_dict['y'][var_test_arr]

        var_pre_obj = fit_torch(CNN, var_src_dict['x_std'][var_srctrain_arr], var_src_dict['y'][var_srctrain_arr], var_ncls_int)
        var_prestate_dict = {k: v.clone() for k, v in var_pre_obj.state_dict().items()}
        var_zeroshot_float = score(var_ytest_arr, predict_torch(var_pre_obj, var_tgt_dict['x_std'][var_test_arr]), var_ncls_int)[1]

        for var_frac_float in var_finetune_fracs_list:
            var_ft_arr = take_fraction(var_tgtpool_arr, var_frac_float, var_seed_int + var_subject_int)
            var_xft_arr, var_yft_arr = var_tgt_dict['x_std'][var_ft_arr], var_tgt_dict['y'][var_ft_arr]
            var_scratch_obj = fit_torch(CNN, var_xft_arr, var_yft_arr, var_ncls_int)
            var_init_obj = CNN(var_tgt_dict['n_ch'], var_ncls_int).to(var_device_obj)
            var_init_obj.load_state_dict(var_prestate_dict)
            var_tuned_obj = fit_torch(CNN, var_xft_arr, var_yft_arr, var_ncls_int, var_init_model_obj=var_init_obj)
            var_row_dict = {"subject": var_subject_int, "fraction": var_frac_float, "n_finetune": len(var_ft_arr),
                            "zero_shot": var_zeroshot_float,
                            "scratch": score(var_ytest_arr, predict_torch(var_scratch_obj, var_tgt_dict['x_std'][var_test_arr]), var_ncls_int)[1],
                            "pretrained": score(var_ytest_arr, predict_torch(var_tuned_obj, var_tgt_dict['x_std'][var_test_arr]), var_ncls_int)[1]}
            var_rows_list.append(var_row_dict)
            print(f"  subject {var_subject_int:>2}  {int(var_frac_float * 100):>3}% target data (n={len(var_ft_arr):>5})  "
                  f"zero-shot {var_zeroshot_float:.3f}  scratch {var_row_dict['scratch']:.3f}  pre-trained {var_row_dict['pretrained']:.3f}")
        pd.DataFrame(var_rows_list).to_csv(f"{var_resultsdir_txt}/E_transfer_{var_source_txt}_to_{var_target_txt}.csv", index=False)

    var_res_df = pd.DataFrame(var_rows_list)
    var_sum_df = var_res_df.groupby('fraction')[['zero_shot', 'scratch', 'pretrained']].agg(['mean', 'std']).round(3)
    print("\nLOSO macro F1 on the target position (mean and sd over 10 people)\n", var_sum_df.to_string())
    for var_frac_float, var_g_df in var_res_df.groupby('fraction'):
        print(f"  {int(var_frac_float * 100)}%: pre-training better on {(var_g_df['pretrained'] > var_g_df['scratch']).sum()} of 10 people")
        append_log({"experiment": "E_transfer", "position": var_target_txt,
                    "model": f"CNN-{var_source_txt}-pretrained-{int(var_frac_float * 100)}pct",
                    "loso_macro_f1_mean": round(var_g_df['pretrained'].mean(), 4), "loso_macro_f1_sd": round(var_g_df['pretrained'].std(ddof=0), 4)})
        append_log({"experiment": "E_transfer", "position": var_target_txt,
                    "model": f"CNN-scratch-{int(var_frac_float * 100)}pct",
                    "loso_macro_f1_mean": round(var_g_df['scratch'].mean(), 4), "loso_macro_f1_sd": round(var_g_df['scratch'].std(ddof=0), 4)})
    return var_res_df


var_transfer_df = run_transfer("leg", "pocket")

  subject  1   10% target data (n= 3233)  zero-shot 0.109  scratch 0.432  pre-trained 0.455
  subject  1   25% target data (n= 8083)  zero-shot 0.109  scratch 0.570  pre-trained 0.585
  subject  2   10% target data (n= 3150)  zero-shot 0.083  scratch 0.303  pre-trained 0.313
  subject  2   25% target data (n= 7807)  zero-shot 0.083  scratch 0.433  pre-trained 0.454
  subject  3   10% target data (n= 3250)  zero-shot 0.076  scratch 0.464  pre-trained 0.419
  subject  3   25% target data (n= 8100)  zero-shot 0.076  scratch 0.636  pre-trained 0.614
  subject  4   10% target data (n= 3237)  zero-shot 0.083  scratch 0.444  pre-trained 0.443
  subject  4   25% target data (n= 8087)  zero-shot 0.083  scratch 0.565  pre-trained 0.569
  subject  5   10% target data (n= 3213)  zero-shot 0.072  scratch 0.367  pre-trained 0.349
  subject  5   25% target data (n= 8063)  zero-shot 0.072  scratch 0.534  pre-trained 0.485
  subject  6   10% target data (n= 3200)  zero-shot 0.093  scratch 0.287  pre-tr

## 11. App: all 10 people, with the new set checks
Zip the local `App` folder (it holds `workout_app.py`, `models/` and `cache/`) as `App.zip` and upload it when asked.
For each person the app trains a CNN on the other nine (skipped if that model is already in `App/models`), then logs all five of that person's sessions.
It reports how many logged sets are right, how many wrong sets a flag catches (and how many of those the new set-length and movement checks catch), how many correct sets get flagged anyway, and how many real sets are missed. About 9 trainings, roughly 20 to 30 minutes on a T4.

In [19]:
!sed -i 's/map_location=var_device_obj)/map_location=var_device_obj, weights_only=False)/' /content/App/workout_app.py
if not os.path.exists('/content/App/workout_app.py'):
    var_upload_dict = files.upload()  # App.zip
    var_zipname_txt = next(iter(var_upload_dict))
    !unzip -q -o "/content/{var_zipname_txt}" -d /content/
!ls /content/App /content/App/models
!cd /content/App && python workout_app.py --data /content/recgym/RecGym.csv evaluate-all
!cd /content/App && python workout_app.py --data /content/recgym/RecGym.csv log --subject 6 --session 2
!mkdir -p {var_resultsdir_txt}/app && cp /content/App/output/all_subjects_*.csv /content/App/output/subject6_session2_* {var_resultsdir_txt}/app/

/content/App:
cache  models  output  workout_app.py

/content/App/models:
wrist_cnn_holdout1.pt  wrist_cnn_holdout6.pt  wrist_CNN_loso_results.npz
Loading cached windows: /content/App/cache/wrist_w80_s40.npz
  subject  1 session 1: 30/31 sets correct, 1 wrong (1 flagged), 3 missed, window macro F1 0.827
  subject  1 session 2: 13/13 sets correct, 0 wrong (0 flagged), 4 missed, window macro F1 0.790
  subject  1 session 3: 24/24 sets correct, 0 wrong (0 flagged), 3 missed, window macro F1 0.867
  subject  1 session 4: 17/17 sets correct, 0 wrong (0 flagged), 3 missed, window macro F1 0.854
  subject  1 session 5: 20/20 sets correct, 0 wrong (0 flagged), 1 missed, window macro F1 0.893
Loading cached windows: /content/App/cache/wrist_w80_s40.npz
Training CNN on 31,968 windows, subject 2 held out, device cuda
  epoch 10/40
  epoch 20/40
  epoch 30/40
  epoch 40/40
Saved model to /content/App/models/wrist_cnn_holdout2.pt
Loading cached windows: /content/App/cache/wrist_w80_s40.npz
  subjec

## 12. Summary
Reads only saved files. If `results_final` from the milestone run is in the same folder (it is when Drive was used for the final notebook), it is used for the comparisons; otherwise only the new results are shown.

In [20]:
var_ext_df = pd.read_csv(var_logpath_txt).sort_values('timestamp').groupby(['experiment', 'position', 'model']).tail(1)
var_cols_list = [c for c in ['experiment', 'position', 'model', 'random_macro_f1', 'blocked_macro_f1',
                             'loso_macro_f1_mean', 'loso_macro_f1_sd', 'gap_macro_f1'] if c in var_ext_df.columns]
print("All new runs (latest of each)")
print(var_ext_df[var_cols_list].to_string(index=False))

var_finallog_txt = f"{var_basedir_txt}/results_final/results_log.csv"
if os.path.exists(var_finallog_txt):
    var_final_df = pd.read_csv(var_finallog_txt).sort_values('timestamp').groupby(['position', 'model']).tail(1)
    var_blocked_df = var_ext_df[var_ext_df['experiment'] == 'B_blocked'][['position', 'model', 'blocked_macro_f1']]
    var_dec_df = var_final_df.merge(var_blocked_df, on=['position', 'model'])
    var_dec_df['overlap_part'] = var_dec_df['random_macro_f1'] - var_dec_df['blocked_macro_f1']
    var_dec_df['person_part'] = var_dec_df['blocked_macro_f1'] - var_dec_df['loso_macro_f1_mean']
    print("\nExp B: where the random-split gap comes from (macro F1 points x 100)")
    print((var_dec_df.set_index(['position', 'model'])[['random_macro_f1', 'blocked_macro_f1', 'loso_macro_f1_mean',
                                                         'overlap_part', 'person_part']]).round(3).to_string())
else:
    print("\nresults_final not found next to results_ext: comparisons with the milestone run will be done offline.")

All new runs (latest of each)
 experiment position                    model  random_macro_f1  blocked_macro_f1  loso_macro_f1_mean  loso_macro_f1_sd  gap_macro_f1
A_sameinput    wrist             CNN-RawInput           0.9651               NaN              0.9432            0.0240        0.0219
A_sameinput    wrist        Keras-StdFeatures           0.9260               NaN              0.9044            0.0197        0.0216
  B_blocked    wrist                      CNN              NaN            0.9584                 NaN               NaN           NaN
  B_blocked    wrist                     LSTM              NaN            0.9357                 NaN               NaN           NaN
  B_blocked    wrist           Keras-Features              NaN            0.9317                 NaN               NaN           NaN
  B_blocked      leg                      CNN              NaN            0.9224                 NaN               NaN           NaN
  B_blocked      leg           Keras-Fe

## 13. Download results
Zips `results_ext` (including the app results in `results_ext/app`) and `figures_ext`; the cache is left out. Send this zip back for the deck and report.

In [21]:
import zipfile

var_zippath_txt = "/content/paiml2_results_ext.zip"
with zipfile.ZipFile(var_zippath_txt, "w") as var_zip_obj:
    for var_folder_txt in [var_resultsdir_txt, var_figuresdir_txt]:
        for var_root_txt, _, var_names_list in os.walk(var_folder_txt):
            for var_name_txt in var_names_list:
                var_path_txt = os.path.join(var_root_txt, var_name_txt)
                var_zip_obj.write(var_path_txt, os.path.relpath(var_path_txt, var_basedir_txt))
files.download(var_zippath_txt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>